# STF Voice Lab — local JARVIS/FRIDAY-style voice study (offline)

**System context.** This lab produces the *voice layer* for the STF advanced system: the energy-optimization agent, the predictive-maintenance agent, and the copilot all speak to the user through this pipeline. The goal is not "a cool voice" — it is a **measured, swappable TTS interface** with per-agent voice presets, latency and loudness budgets, that later swaps from local Kokoro to ElevenLabs (or any engine) without touching the orchestrator.

Design rules baked into every cell below:

1. **Style, not clone.** We synthesize an *original* voice "in the style of" a crisp, measured AI assistant. We never train or clone on ripped movie audio, and nothing here ships to the deployed site (licensed voices only, there).
2. **Presets are data.** Voice = blend string + speed + a parameterized DSP chain, stored as JSON. Copilot / watchdog / energy agent each get their own preset.
3. **Everything is measured.** Every candidate reports synthesis latency, real-time factor, and LUFS. Interactive SLO: first audio < 300 ms.
4. **Offline.** After the one-time model download, nothing leaves the machine.

## Legal / safety boundary (read once)

- JARVIS / FRIDAY are performances by real actors. Cloning those voiceprints — from film rips or impersonator samples — is the line we do not cross. Voiceprints of real people carry personality rights; the source audio is copyrighted.
- What we do instead: pick stock open-license voices, **blend** them into an original timbre, and shape the *feel* with a DSP chain (EQ, compression, reverb). Style is not protected; a person's voice is.
- All output stays local. The deployed site uses licensed voices (ElevenLabs voice library or a recorded original) via the same engine interface shown at the end.

In [7]:
# --- Setup: run once, then comment the install line back out ---
# %pip install -q kokoro soundfile pedalboard pyloudnorm

import os, json, time, shutil, subprocess
from pathlib import Path
import numpy as np

# If we are inside the repo, keep everything under voice-lab/
ROOT = Path(".") if Path.cwd().name == "voice-lab" else Path("voice-lab")
OUT = ROOT / "outputs"
(OUT / "raw").mkdir(parents=True, exist_ok=True)
(OUT / "processed").mkdir(parents=True, exist_ok=True)
(OUT / "rig").mkdir(parents=True, exist_ok=True)
PRESETS = ROOT / "presets"
PRESETS.mkdir(parents=True, exist_ok=True)

# Kokoro uses espeak-ng for English grapheme-to-phoneme. macOS: brew install espeak-ng
if shutil.which("espeak-ng") is None:
    print("WARNING: espeak-ng not found. Run  brew install espeak-ng  then restart the kernel.")
else:
    print("espeak-ng OK ->", shutil.which("espeak-ng"))

SAMPLE_RATE = 24000

# Factory-flavored lines: these are the kinds of utterances the energy/PdM agents
# will actually speak, so we tune the voice on realistic content.
LINES = [
    "Good morning. All systems are running within normal parameters.",
    "Order 12 is now baking in zone two at 220 degrees.",
    "Energy use is up 8 percent this cycle. I have shifted the next batch to off peak.",
    "The vacuum gripper is showing early wear. Maintenance is recommended within 40 cycles.",
    "Backpressure detected at the tunnel gate. Holding carrier intake for 15 seconds.",
]
print("setup done")


espeak-ng OK -> /opt/homebrew/bin/espeak-ng
setup done


## Why Kokoro, and why *blending*

- **Kokoro-82M**: Apache 2.0 (no ceiling if this ever ships in a product), ~350 MB, runs 10x+ real time on Apple Silicon, sub-200 ms latency — comfortably inside our 300 ms interactive budget.
- **Blending**: Kokoro accepts `voice="bm_george#3+bm_lewis#1"` — weighted mixes of its preset voice embeddings. That turns "find the JARVIS voice" into a *systematic search over a blend grid*, which is exactly what makes this a study instead of a vibe.
- Voice name prefix: `a`=American, `b`=British, `f/m`=female/male. We lean British male for the JARVIS register; `am_` voices are controls.

In [8]:
import torch
from kokoro import KPipeline
import soundfile as sf
from IPython.display import Audio, display

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)
try:
    pipeline = KPipeline(lang_code="b", device=device)   # b = British English g2p
except TypeError:  # older kokoro builds: no device kwarg
    pipeline = KPipeline(lang_code="b")

def synth(text, voice="bm_george", speed=1.0):
    """Synthesize text -> (mono float32 numpy @ 24 kHz, synth time in ms)."""
    t0 = time.perf_counter()
    chunks = []
    for gs, ps, audio in pipeline(text, voice=voice, speed=speed):
        chunks.append(audio.detach().cpu().float().numpy())
    ms = (time.perf_counter() - t0) * 1000
    if not chunks:
        raise RuntimeError("no audio produced")
    x = np.concatenate(chunks)
    if x.ndim > 1:
        x = x.mean(axis=0)  # safety: some builds return (2, T)
    return x.astype(np.float32), ms

# smoke test
x, ms = synth(LINES[0], voice="bm_george", speed=0.95)
print(f"smoke test: {len(x)/SAMPLE_RATE:.2f}s audio, synth {ms:.0f} ms")
display(Audio(x, rate=SAMPLE_RATE))

device: mps
smoke test: 5.08s audio, synth 900 ms


## Step 1 — the blend space

Five candidate timbres. `A_control` is an American voice to anchor the scale; the `J*` blends aim cool/deliberate (JARVIS register); `F1` aims warmer/faster (FRIDAY register). Speed is tuned later — first we fix the timbre.

In [9]:
BLENDS = {
    "A_control": "am_michael",
    "B_base":    "bm_george",
    "J1_cool":   "bm_george#3+bm_lewis#1",
    "J2_dark":   "bm_lewis#2+bm_daniel#1",
    "F_base":    "bf_emma",
    "F1_warm":   "bf_emma#2+af_heart#1",
    "F2_lead":   "bf_isabella#2+bf_emma#1",
}

AUDITION_TEXT = LINES[0] + " " + LINES[2]

raw_files = {}
for name, voice in BLENDS.items():
    try:
        x, ms = synth(AUDITION_TEXT, voice=voice, speed=0.95)
    except Exception as e:
        print(f"{name:10s} FAILED: {e}")
        continue
    p = OUT / "raw" / f"{name}.wav"
    sf.write(p, x, SAMPLE_RATE)
    raw_files[name] = p
    print(f"{name:10s} voice={voice:28s} audio={len(x)/SAMPLE_RATE:4.1f}s synth={ms:4.0f} ms -> {p}")

for name, p in raw_files.items():
    print(f"\n### {name} ({BLENDS[name]})")


A_control  voice=am_michael                   audio=11.0s synth=1083 ms -> outputs/raw/A_control.wav
B_base     voice=bm_george                    audio=10.6s synth= 703 ms -> outputs/raw/B_base.wav
J1_cool    voice=bm_george#3+bm_lewis#1       audio=10.5s synth= 822 ms -> outputs/raw/J1_cool.wav
J2_dark    voice=bm_lewis#2+bm_daniel#1       audio= 9.9s synth= 774 ms -> outputs/raw/J2_dark.wav
F_base     voice=bf_emma                      audio= 9.3s synth= 764 ms -> outputs/raw/F_base.wav
F1_warm    voice=bf_emma#2+af_heart#1         audio= 9.5s synth= 748 ms -> outputs/raw/F1_warm.wav
F2_lead    voice=bf_isabella#2+bf_emma#1      audio= 9.2s synth= 724 ms -> outputs/raw/F2_lead.wav

### A_control (am_michael)

### B_base (bm_george)

### J1_cool (bm_george#3+bm_lewis#1)

### J2_dark (bm_lewis#2+bm_daniel#1)

### F_base (bf_emma)

### F1_warm (bf_emma#2+af_heart#1)

### F2_lead (bf_isabella#2+bf_emma#1)


## Step 2 — the DSP chain (where "JARVIS" actually lives)

The film sound is *not* a robot voice: it is a close-mic, natural delivery with restrained processing. Reproduced as parameters:

| Stage | Effect | Why |
|---|---|---|
| highpass @ ~70 Hz | rumble removal | keeps speech forward |
| compressor ~2.5:1 | even dynamics | authority = consistency |
| high-shelf +2..3 dB @ ~4 kHz | presence lift | the "crisp AI" quality |
| chorus, tiny mix | faint doubling | the "computer beneath the voice" |
| reverb, small room, ~10% wet | short ambience | not dry-clinical, not cavernous |
| limiter | peak control | safe playback level |

Two presets: **JARVIS_V1** (cooler, speed 0.92, deliberate) and **FRIDAY_V1** (warmer, speed 1.0, lighter chain). The presets are plain dicts — diffable, versionable.

In [10]:
import pedalboard as pb
from pedalboard import Pedalboard

JARVIS_V1 = {
    "voice": "J1_cool", "speed": 0.92,
    "chain": [
        {"name": "HighpassFilter", "cutoff_frequency_hz": 70},
        {"name": "Compressor", "threshold_db": -20, "ratio": 2.5, "attack_ms": 6, "release_ms": 150},
        {"name": "HighShelfFilter", "cutoff_frequency_hz": 4200, "gain_db": 3.0, "q": 0.7},
        {"name": "Chorus", "rate_hz": 0.7, "depth": 0.08, "mix": 0.05},
        {"name": "Reverb", "room_size": 0.12, "damping": 0.85, "wet_level": 0.10, "dry_level": 0.95, "width": 0.4},
        {"name": "Limiter", "threshold_db": -1.5},
    ],
}


FRIDAY_V1 = {
    "voice": "F1_warm", "speed": 1.00,
    "chain": [
        {"name": "HighpassFilter", "cutoff_frequency_hz": 60},
        {"name": "Compressor", "threshold_db": -18, "ratio": 2.0, "attack_ms": 5, "release_ms": 120},
        {"name": "HighShelfFilter", "cutoff_frequency_hz": 4500, "gain_db": 2.0, "q": 0.7},
        {"name": "Chorus", "rate_hz": 0.9, "depth": 0.10, "mix": 0.08},
        {"name": "Reverb", "room_size": 0.18, "damping": 0.80, "wet_level": 0.13, "dry_level": 0.95, "width": 0.5},
        {"name": "Limiter", "threshold_db": -1.5},
    ],
}

def build_board(spec):
    return Pedalboard([
        getattr(pb, c["name"])(**{("cutoff_frequency_hz" if k == "cutoff_hz_hz" else k): v for k, v in c.items() if k != "name"})
        for c in spec["chain"]
    ])

def process_file(wav_in, wav_out, spec):
    x, sr = sf.read(str(wav_in))
    if x.ndim > 1:
        x = x.mean(axis=1)
    t0 = time.perf_counter()
    y = build_board(spec)(x.astype(np.float32), sr)
    ms = (time.perf_counter() - t0) * 1000
    sf.write(str(wav_out), y, sr)
    return ms

# Render the A/B candidates: raw control + 3 processed personalities
CANDIDATES = {
    "raw_control": ("A_control", None,      1.00),
    "jarvis_J1":   ("J1_cool",   JARVIS_V1, 0.92),
    "jarvis_J2":   ("J2_dark",   JARVIS_V1, 0.92),
    "friday_F1":   ("F1_warm",   FRIDAY_V1, 1.00),
}

rig_files = {}
for cname, (blend, spec, speed) in CANDIDATES.items():
    x, _ = synth(AUDITION_TEXT, voice=BLENDS[blend], speed=speed)
    raw_p = OUT / "rig" / f"{cname}_raw.wav"
    sf.write(raw_p, x, SAMPLE_RATE)
    if spec is None:
        final_p = raw_p
        print(f"{cname:12s} raw (no chain)")
    else:
        final_p = OUT / "rig" / f"{cname}.wav"
        pms = process_file(raw_p, final_p, spec)
        print(f"{cname:12s} chain {pms:5.1f} ms -> {final_p.name}")
    rig_files[cname] = final_p


raw_control  raw (no chain)
jarvis_J1    chain  20.6 ms -> jarvis_J1.wav
jarvis_J2    chain  19.5 ms -> jarvis_J2.wav
friday_F1    chain  18.6 ms -> friday_F1.wav


## Step 3 — the A/B listening rig

Shuffled order (reduces order bias). For each candidate: it plays, you score four attributes 1-5 (`q` quits). Results land in `outputs/rig/scores.csv`. Score with headphones, and judge *for the factory use-case*: a voice you love for 30 seconds may grate at cycle #200 — that is what the fatigue axis is for.

In [11]:
import pandas as pd

ATTRS = [
    ("clarity",   "1 = muddy, 5 = crystal"),
    ("authority", "1 = casual, 5 = commanding"),
    ("ai_feel",   "1 = plain human, 5 = perfect AI assistant"),
    ("fatigue",   "1 = could listen all day, 5 = annoying"),
]

rng = np.random.RandomState(7)
order = list(rig_files.keys())
order = [order[i] for i in rng.permutation(len(order))]  # shuffled to reduce bias

rows = []
try:
    for cname in order:
        print(f"\n===== {cname} =====")
        display(Audio(str(rig_files[cname])))
        row = {"candidate": cname}
        for attr, desc in ATTRS:
            while True:
                v = input(f"{attr} ({desc}), 1-5, q=quit: ").strip().lower()
                if v == "q":
                    raise KeyboardInterrupt
                if len(v) == 1 and v in "12345":
                    row[attr] = int(v)
                    break
        rows.append(row)
except KeyboardInterrupt:
    print("stopped early - partial scores kept")

df = pd.DataFrame(rows)
if len(df):
    df.to_csv(OUT / "rig" / "scores.csv", index=False)
    display(df)
    print("\nmeans:")
    display(df.drop(columns=["candidate"]).mean().round(2).to_frame("mean"))


===== jarvis_J2 =====


stopped early - partial scores kept


## Step 4 — measure, then write the voice card

Numbers, not vibes: LUFS (loudness consistency across candidates), synthesis latency, real-time factor. The **voice card** is the artifact the STF backend will load — per-agent preset mapping + measured budgets. Edit `selected` after your listening rig.

In [12]:
if "SAMPLE_RATE" not in globals():
    SAMPLE_RATE = 24000
import pyloudnorm as pyln

meter = pyln.Meter(SAMPLE_RATE)
measured = {}
for cname, p in rig_files.items():
    x, sr = sf.read(str(p))
    if x.ndim > 1:
        x = x.mean(axis=1)
    blend, spec, speed = CANDIDATES[cname]
    _, sms = synth(AUDITION_TEXT, voice=BLENDS[blend], speed=speed)
    dur = len(x) / sr
    measured[cname] = {
        "lufs": round(float(meter.integrated_loudness(x)), 1),
        "synth_ms": round(sms, 0),
        "dur_s": round(dur, 2),
        "rtf": round(sms / 1000.0 / dur, 2),
    }
mdf = pd.DataFrame(measured).T
display(mdf)

card = {
    "engine": "kokoro-82m-local",
    "selected": "jarvis_J1",          # <- edit after the A/B rig
    "presets": {"jarvis_v1": JARVIS_V1, "friday_v1": FRIDAY_V1},
    "agents": {"copilot": "friday_v1", "watchdog": "jarvis_v1", "energy_agent": "jarvis_v1"},
    "measurements": measured,
    "budget": {"first_audio_ms_target": 300, "chain_ms_max": 50},
}
card_path = PRESETS / "voice_card.json"
card_path.write_text(json.dumps(card, indent=2))


,lufs,synth_ms,dur_s,rtf
raw_control,-27.2,738.0,10.68,0.07
jarvis_J1,-16.5,655.0,10.75,0.06
jarvis_J2,-17.6,682.0,10.20,0.07
friday_F1,-14.8,609.0,9.10,0.07


2497

## Step 5 — the swappable engine interface (how this plugs into STF)

The energy-optimization and PdM orchestrator never calls Kokoro directly — it calls a `TTSEngine`. Local development uses `LocalKokoroTTS` (this lab); the deployed site uses `ElevenLabsTTS` (licensed voices, streaming). Same signature, zero orchestrator changes — the property that makes the voice layer boring to integrate and safe to swap.

In [13]:
from typing import Protocol, Dict

class TTSEngine(Protocol):
    """One method. That is the whole integration surface."""
    def synth_to_wav(self, text: str, preset_name: str) -> bytes: ...

class LocalKokoroTTS:
    """Dev engine: this lab. Kokoro + pedalboard presets, fully offline."""
    def __init__(self, voice_card: dict):
        self.card = voice_card
    def synth_to_wav(self, text: str, preset_name: str) -> bytes:
        spec = self.card["presets"][preset_name]
        x, _ = synth(text, voice=BLENDS[spec["voice"]], speed=spec["speed"])
        y = build_board(spec)(x, SAMPLE_RATE)
        return (y * 32767).astype(np.int16).tobytes()  # raw PCM; wrap in WAV header in prod

class ElevenLabsTTS:
    """Deployed engine. Same interface, streaming websocket, licensed voices only."""
    def synth_to_wav(self, text: str, preset_name: str) -> bytes:
        # map preset_name -> (voice_id, model, stability) from voice_card
        # stream chunks over the websocket; honor the same first-audio budget
        raise NotImplementedError("deployed only; needs ELEVENLABS_API_KEY")

# Which voice speaks for which agent (multi-agent voice design):
AGENT_PRESETS = {"copilot": "friday_v1", "watchdog": "jarvis_v1", "energy_agent": "jarvis_v1"}

tts = LocalKokoroTTS(card)
print("engine ready; agent->preset:", AGENT_PRESETS)

engine ready; agent->preset: {'copilot': 'friday_v1', 'watchdog': 'jarvis_v1', 'energy_agent': 'jarvis_v1'}


## Next steps (toward the energy + PdM system)

1. Tune winners: grid search speed in 0.88-0.98 and shelf gain in 2-4 dB around the winning candidate; re-run the rig only on the top-2.
2. Freeze the voice card; commit it as data (not code) so presets are reviewable diffs.
3. Wire `LocalKokoroTTS` behind the FastAPI chat endpoint as a dev flag; deployed builds fall through to ElevenLabsTTS.
4. Watchdog/energy alerts: measure *perceived* latency end-to-end (event -> speech) — that number goes in the system SLO table next to failover and gate latency.